# Задача 1. Two Sum

**Условие.** Дан массив целых чисел `arr` и целое число `k`. Нужно найти индексы двух разных элементов, сумма которых равна `k`. Гарантируется, что такая пара ровно одна.

Индексы отсчитываются с нуля. Ответ выводится через пробел в порядке возрастания.

- `arr = [1, 3, 4, 10]`, `k = 7` → `1 2`.
- `arr = [5, 5, 1, 4]`, `k = 10` → `0 1`.

## Идея решения

Используем словарь `dictionary`: ключ — уже просмотренное число, значение — его индекс. Для текущего числа `arr[i]` вычисляем недостающее слагаемое `complement = k - arr[i]`.

1. Если `complement` есть в словаре, возвращаем его индекс и текущий индекс `i`.
2. Иначе записываем текущее число и его индекс в словарь и переходим дальше.

Проверка выполняется **до добавления** текущего числа: один элемент не может использоваться дважды. При этом два одинаковых числа на разных позициях допустимы.

Решение основано на исходном коде со словарём и циклом `while`. Вместо печати найденных индексов внутри цикла функция возвращает пару и сразу завершает работу. Первым возвращается сохранённый индекс: он всегда меньше текущего. Проверка `value in dictionary` проверяет ключи, поэтому вызов `.keys()` не нужен.


In [1]:
def two_sum(arr: list[int], k: int) -> tuple[int, int]:
    """Вернуть индексы двух элементов с суммой k в порядке возрастания."""
    dictionary = {}
    i = 0

    while i < len(arr):
        complement = k - arr[i]
        if complement in dictionary:
            return dictionary[complement], i

        dictionary[arr[i]] = i
        i += 1

    # По условию задачи сюда не попадём.
    raise ValueError("Пара элементов с заданной суммой не найдена")


## Почему алгоритм корректен

Перед каждой итерацией словарь хранит значения из уже просмотренной части массива и соответствующие им индексы. Если значение встречалось несколько раз, достаточно хранить один из его индексов.

Если `complement` найден, то `arr[j] = k - arr[i]`, где `j = dictionary[complement]`. Следовательно, `arr[j] + arr[i] = k`. Так как в словаре только ранее просмотренные элементы, `j < i`: индексы различны и сразу расположены по возрастанию.

Пусть искомая пара имеет индексы `p < q`. Если функция ещё не завершилась, при обработке `q` значение `arr[p]` уже есть в словаре. Поэтому нужное дополнение будет найдено. По условию допустимая пара ровно одна, значит функция возвращает именно её.

## Оценка сложности

Пусть `n = len(arr)`.

- **Время: O(n)** в среднем. Просматриваем массив один раз; поиск и запись в словарь занимают O(1) в среднем. При патологических коллизиях хешей теоретическая худшая оценка — O(n²).
- **Дополнительная память: O(n)**. В словаре хранится не более `n` различных значений и их индексов.

## Пошаговая визуализация

Возьмём `arr = [6, -3, 0, 8, 9, 2]`, `k = 5`. Здесь есть отрицательное число и ноль. Единственная пара — `-3 + 8 = 5`, индексы `1 3`.

Словарь показан **до** проверки текущего элемента.

| Шаг | Индекс `i` | `arr[i]` | Нужно `k - arr[i]` | Словарь до проверки | Действие |
| --- | --- | --- | --- | --- | --- |
| 1 | 0 | 6 | -1 | `{}` | -1 нет → записываем `6: 0` |
| 2 | 1 | -3 | 8 | `{6: 0}` | 8 нет → записываем `-3: 1` |
| 3 | 2 | 0 | 5 | `{6: 0, -3: 1}` | 5 нет → записываем `0: 2` |
| 4 | 3 | 8 | -3 | `{6: 0, -3: 1, 0: 2}` | -3 есть с индексом 1 → возвращаем `(1, 3)` |

Элементы с индексами 4 и 5 не обрабатываются: ответ уже найден.


In [3]:
arr = [6, -3, 0, 8, 9, 2]
k = 5
print(*two_sum(arr, k))


1 3


## Тесты

Проверяем примеры из условия, минимальный размер массива, повторяющиеся значения, нули, отрицательные числа, пару на краях, большие целые числа и случай, когда текущее число равно половине цели, но использовать его дважды нельзя.

Дополнительно проверяем выбранное поведение за пределами условия: если пары нет, функция должна вызвать `ValueError`.


In [4]:
test_cases = [
    ("Первый пример", [1, 3, 4, 10], 7, (1, 2)),
    ("Второй пример: одинаковые числа", [5, 5, 1, 4], 10, (0, 1)),
    ("Минимальный размер", [2, 7], 9, (0, 1)),
    ("Два нуля", [0, 0], 0, (0, 1)),
    ("Отрицательная цель", [-8, -3, -1, 6], -11, (0, 1)),
    ("Смешанные знаки и нулевая цель", [-7, 4, 7, 9], 0, (0, 2)),
    ("Ноль и число, равное цели", [0, 4, 9], 4, (0, 1)),
    ("Пара на краях массива", [8, 1, 2, 3, 12], 20, (0, 4)),
    ("Нельзя использовать один элемент дважды", [3, 2, 4], 6, (1, 2)),
    ("Повторы вне искомой пары", [1, 1, 3, 8], 11, (2, 3)),
    ("Большие целые числа", [10**20, 5, -(10**20) + 7], 7, (0, 2)),
    ("Пример из визуализации", [6, -3, 0, 8, 9, 2], 5, (1, 3)),
]

for name, arr, k, expected in test_cases:
    original = arr.copy()
    result = two_sum(arr, k)
    assert result == expected, f"{name}: ожидалось {expected}, получено {result}"
    i, j = result
    assert 0 <= i < j < len(arr), name
    assert arr[i] + arr[j] == k, name
    assert arr == original, f"{name}: функция изменила массив"

invalid_cases = [([], 0), ([3], 6), ([1, 2, 4], 10)]
for arr, k in invalid_cases:
    try:
        two_sum(arr, k)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Для {arr}, k={k} ожидался ValueError")

print(f"Все тесты пройдены: {len(test_cases)} с парой и {len(invalid_cases)} без пары.")


Все тесты пройдены: 12 с парой и 3 без пары.
